# Avito: кандидатогенерация для поиска услуг

**Задача:** по поисковому запросу отобрать до 50 объявлений-кандидатов
из корпуса, которые дальше пойдут на этап ранжирования.

**Метрика:** Recall@50 — усреднённая по запросам доля релевантных
объявлений, попавших в топ-50.

**Подход:**
лексический поиск (TF-IDF по словам + символьным n-граммам) с косинусной
близостью запрос↔объявление, плюс эвристический буст за совпадение
категории/локации.

In [68]:
import pandas as pd
import numpy as np

pd.set_option("display.max_colwidth", 120)  # чтобы длинные тексты не обрезались в выводе

In [69]:
TRAIN_PATH = "data/train.parquet"
BENCH_QUERIES_PATH = "data/benchmark_queries.parquet"
BENCH_ITEMS_PATH = "data/benchmark_items.parquet"

train = pd.read_parquet(TRAIN_PATH)
bench_queries = pd.read_parquet(BENCH_QUERIES_PATH)
bench_items = pd.read_parquet(BENCH_ITEMS_PATH)

print("train:", train.shape)
print("bench_queries:", bench_queries.shape)
print("bench_items:", bench_items.shape)

train: (497673, 19)
bench_queries: (2452, 6)
bench_items: (189212, 14)


In [70]:
print(train.dtypes)

search_query                     str
search_location_id             int64
search_is_delivery_search      int32
search_infm_params_text          str
search_category                int64
item_title_raw                   str
item_rating_reviews_count    float64
item_rating                  float64
item_price                    object
item_microcat_id               int64
item_longitude                object
item_location_id               int64
item_latitude                 object
item_is_phone_hidden            bool
item_is_message_forbidden       bool
item_infm_params_text            str
item_id                          str
item_description_raw             str
item_category_id               int64
dtype: object


In [71]:
train.head(3).T  # .T — транспонируем, чтобы длинные тексты было видно построчно

,0,1,2
search_query,скупка телевизоров,автоподбор,баня на дровах
search_location_id,652430,640860,653240
search_is_delivery_search,0,0,0
search_infm_params_text,,Рейтинг пользователя 4 звезды и выше,"Онлайн-запись Тип услуги СПА-услуги, массаж Вид услуги Красота, здоровье"
search_category,114,114,114
item_title_raw,Скупка б/у техники,Автоподбор Разовый осмотр автомобиля,"Баня на дровах ""Прованс"" на Цветочной"
item_rating_reviews_count,91.0,462.0,NaN
item_rating,4.989011,4.982684,NaN
item_price,1.000000000000000,3500.000000000000000,1600.000000000000000
item_microcat_id,2303374,2303374,86469


## 2. Пропуски и качество данных

In [72]:
print("Пропуски в train:")
print(train.isna().mean().sort_values(ascending=False))

print("\nПропуски в bench_items:")
print(bench_items.isna().mean().sort_values(ascending=False))

print("\nПропуски в bench_queries:")
print(bench_queries.isna().mean().sort_values(ascending=False))

Пропуски в train:
item_rating                  0.056728
item_rating_reviews_count    0.038602
item_longitude               0.000008
item_latitude                0.000008
search_is_delivery_search    0.000000
search_query                 0.000000
search_location_id           0.000000
item_title_raw               0.000000
search_category              0.000000
item_price                   0.000000
search_infm_params_text      0.000000
item_microcat_id             0.000000
item_location_id             0.000000
item_is_phone_hidden         0.000000
item_is_message_forbidden    0.000000
item_infm_params_text        0.000000
item_id                      0.000000
item_description_raw         0.000000
item_category_id             0.000000
dtype: float64

Пропуски в bench_items:
item_rating                  0.093181
item_rating_reviews_count    0.061386
item_description_raw         0.000185
item_longitude               0.000005
item_latitude                0.000005
item_title_raw               0

## 3. Реконструкция понятия "запрос" в train

В train.parquet нет `query_id` — есть только пары запрос-объявление.
Чтобы посчитать recall на валидации, нужно понять: что считать
"одним и тем же запросом"? Логично — уникальная комбинация всех
`search_*`-полей (текст + локация + доставка + фильтры + категория).
Тогда для одного и того же запроса может быть несколько релевантных
объявлений (если пользователь искал одно и то же несколько раз и
выбирал разные объявления).

In [73]:
search_cols = [c for c in train.columns if c.startswith("search_")]
item_cols = [c for c in train.columns if c.startswith("item_")]
print("Колонки запроса:", search_cols)
print("Колонки объявления:", item_cols)

n_unique_queries = train[search_cols].drop_duplicates().shape[0]
print(f"\nВсего строк в train: {len(train)}")
print(f"Уникальных комбинаций search_*-полей (т.е. 'запросов'): {n_unique_queries}")

# сколько релевантных объявлений приходится на один такой "запрос"
rel_counts = train.groupby(search_cols, dropna=False)["item_id"].nunique()
print("\nРаспределение числа релевантных item_id на один запрос:")
print(rel_counts.describe())
print("\nТоп значений:")
print(rel_counts.value_counts().head(10))

Колонки запроса: ['search_query', 'search_location_id', 'search_is_delivery_search', 'search_infm_params_text', 'search_category']
Колонки объявления: ['item_title_raw', 'item_rating_reviews_count', 'item_rating', 'item_price', 'item_microcat_id', 'item_longitude', 'item_location_id', 'item_latitude', 'item_is_phone_hidden', 'item_is_message_forbidden', 'item_infm_params_text', 'item_id', 'item_description_raw', 'item_category_id']

Всего строк в train: 497673
Уникальных комбинаций search_*-полей (т.е. 'запросов'): 354463

Распределение числа релевантных item_id на один запрос:
count    354463.000000
mean          1.317638
std           1.611180
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max         278.000000
Name: item_id, dtype: float64

Топ значений:
item_id
1     301297
2      33624
3       9343
4       3922
5       2019
6       1188
7        737
8        543
9        388
10       255
Name: count, dtype: int64


In [74]:
print("Уникальных item_id в train:", train["item_id"].nunique())
print("Уникальных item_id в bench_items:", bench_items["item_id"].nunique())

# проверим, что для одного item_id признаки объявления в train не "плавают"
# (иначе при дедупликации по item_id придётся решать, какую версию признаков брать)
dup_check = train.groupby("item_id")[item_cols].nunique()
print("\nМакс. число уникальных значений item_*-признаков для одного item_id:")
print(dup_check.max())

Уникальных item_id в train: 344825
Уникальных item_id в bench_items: 189212

Макс. число уникальных значений item_*-признаков для одного item_id:
item_title_raw               1
item_rating_reviews_count    1
item_rating                  1
item_price                   1
item_microcat_id             1
item_longitude               1
item_location_id             1
item_latitude                1
item_is_phone_hidden         1
item_is_message_forbidden    1
item_infm_params_text        1
item_id                      1
item_description_raw         1
item_category_id             1
dtype: int64


In [75]:
for col in ["search_query", "search_infm_params_text"]:
    print(f"--- {col}, примеры ---")
    print(train[col].dropna().sample(10, random_state=42).tolist())
    print()

for col in ["item_title_raw", "item_infm_params_text"]:
    print(f"--- {col}, примеры ---")
    print(train[col].dropna().sample(5, random_state=42).tolist())
    print()

--- search_query, примеры ---
['брови', 'услуги манипулятора', 'ремонт принтеров', 'ремонт кондиционеров', 'передержка попугаев', 'вспашка мотоблоком', 'строительные бригады', 'электрик', 'предрейсовые осмотр', 'подключение интернета в квартиру']

--- search_infm_params_text, примеры ---
['Вид услуги Красота, здоровье', '', 'Тип услуги Принтеры Вид услуги Компьютерная помощь', 'Вид услуги Ремонт и обслуживание техники', 'Вид услуги Уход за животными', '', '', 'Тип услуги Электрика Вид услуги Ремонт и отделка', '', 'Тип услуги Интернет и другие сети Вид услуги Компьютерная помощь']

--- item_title_raw, примеры ---
['Требуется модель на брови', 'Аренда и услуги манипулятора, стрела 14 т, 14 м, борт 22 т, 8.5 м', 'Ремонт принтеров и мфу', 'Обслуживание кондиционеров и сплит system', 'Передержка домашних животных']

--- item_infm_params_text, примеры ---
['Вид услуги Красота, здоровье Место оказания услуг ул. Ватутина Тип услуги Ресницы, брови Тип стоимости за услугу Начальная цена Где вы 

In [76]:
train["search_query_wordcount"] = train["search_query"].fillna("").str.split().str.len()
print(train["search_query_wordcount"].describe())
print("\nСамые частые длины запроса в словах:")
print(train["search_query_wordcount"].value_counts().sort_index().head(10))

count    497673.000000
mean          2.339287
std           1.067237
min           1.000000
25%           2.000000
50%           2.000000
75%           3.000000
max          17.000000
Name: search_query_wordcount, dtype: float64

Самые частые длины запроса в словах:
search_query_wordcount
1      98513
2     222714
3     111829
4      46359
5      12919
6       3411
7       1331
8        494
9         74
10         8
Name: count, dtype: int64


## 4. Пересечение item_id между train и benchmark_items

На данном этапе выведила для себя важный момент: пул объявлений в train и в бенчмарке — это одно и то же
множество (просто наблюдаем часть) или разные срезы каталога?
От этого зависит, насколько валидация на train будет репрезентативна
для реального сабмита.

In [77]:
train_item_ids = set(train["item_id"].unique())
bench_item_ids = set(bench_items["item_id"].unique())

overlap = train_item_ids & bench_item_ids
print(f"Объявлений в train: {len(train_item_ids)}")
print(f"Объявлений в bench_items: {len(bench_item_ids)}")
print(f"Пересечение: {len(overlap)}")
print(f"Доля train-объявлений, которых нет в bench_items: {1 - len(overlap)/len(train_item_ids):.2%}")
print(f"Доля bench_items, которых нет в train: {1 - len(overlap)/len(bench_item_ids):.2%}")

Объявлений в train: 344825
Объявлений в bench_items: 189212
Пересечение: 18142
Доля train-объявлений, которых нет в bench_items: 94.74%
Доля bench_items, которых нет в train: 90.41%


## 5. Схема валидации

TF-IDF — модель без обучения на лейблах (fit по тексту не "видит" правильные
ответы), поэтому классической утечки при её обучении на всех данных нет.
Но чтобы честно оценить recall@50 **до** отправки на платформу, нужно
отложить часть запросов как "валидационные" и не подсматривать в их
правильные ответы при настройке пайплайна.

План:
1. Свернуть train в таблицу "запрос → список релевантных item_id"
   (группировка по search_*-колонкам).
2. Отложить 5% запросов как валидационные (held-out).
3. Индекс объявлений для поиска на валидации строим из **всех**
   объявлений train (дедуплицированных по item_id).

In [78]:
# 1. Сворачиваем train в "запрос -> список релевантных item_id"
queries_df = (
    train.groupby(search_cols, dropna=False)["item_id"]
    .apply(lambda s: list(s.unique()))
    .reset_index()
    .rename(columns={"item_id": "relevant_item_ids"})
)
print("Всего уникальных запросов:", len(queries_df))
queries_df.head(3)

Всего уникальных запросов: 354463


,search_query,search_location_id,search_is_delivery_search,search_infm_params_text,search_category,relevant_item_ids
0,"""промывка радиатора автомобиля без снятия люберцы""",637640,0,,114,[791cc82871ec82d7]
1,"""ремонт ванной под ключ""",637640,0,Тип услуги Ремонт квартир и домов под ключ Вид услуги Ремонт и отделка,114,[87d23913a737eceb]
2,"""установка рулевой колонки на лодку пвх""",107620,0,,114,[9d22e6e798d0ef38]


In [79]:
# 2. Train/val split запросов
from sklearn.model_selection import train_test_split

train_queries, val_queries = train_test_split(
    queries_df, test_size=0.05, random_state=42
)
print("Train queries:", len(train_queries))
print("Val queries:", len(val_queries))

Train queries: 336739
Val queries: 17724


In [80]:
# 3. Индекс объявлений для валидации — все уникальные объявления train
items_index = train.drop_duplicates(subset="item_id")[item_cols].reset_index(drop=True)
print("Объявлений в индексе для валидации:", len(items_index))

Объявлений в индексе для валидации: 344825


## 6. Функция подготовки текста объявления/запроса и метрика recall@50

In [81]:
def build_item_text(df):
    """Склеиваем текстовые поля объявления в один документ для TF-IDF.
    Title дублируем — он самый информативный, так мы увеличиваем его вес
    без введения отдельной модели весов."""
    title = df["item_title_raw"].fillna("")
    infm = df["item_infm_params_text"].fillna("")
    desc = df["item_description_raw"].fillna("")
    return (title + " " + title + " " + infm + " " + desc).str.lower()

def build_query_text(df):
    """Аналогично для запроса: текст запроса + текстовые фильтры."""
    q = df["search_query"].fillna("")
    infm = df["search_infm_params_text"].fillna("")
    return (q + " " + q + " " + infm).str.lower()

In [82]:
def recall_at_k(predictions, relevant):
    """
    predictions: список списков (или множеств) предсказанных item_id, по одному на запрос
    relevant: список списков истинно релевантных item_id, по одному на запрос
    Возвращает средний recall@k по запросам.
    """
    scores = []
    for pred, rel in zip(predictions, relevant):
        pred_set = set(pred)
        rel_set = set(rel)
        if len(rel_set) == 0:
            continue
        scores.append(len(pred_set & rel_set) / len(rel_set))
    return np.mean(scores)

In [ ]:
from scipy.sparse import csr_matrix

def top_k_candidates(query_vecs, item_vecs, item_ids, k=50, batch_size=500):
    """
    query_vecs: sparse TF-IDF матрица запросов (n_queries x vocab)
    item_vecs:  sparse TF-IDF матрица объявлений (n_items x vocab)
    item_ids:   np.array с item_id, соответствующими строкам item_vecs
    Возвращает список списков top-k item_id по косинусной близости
    (TF-IDF векторы уже L2-нормированы sklearn'ом -> dot product = cosine similarity).

    Считаю батчами по запросам, чтобы не взрывать память: результат
    sparse @ sparse.T естественно остаётся разреженным (ненулевые элементы
    только там, где есть общие слова), поэтому беру top-k прямо из
    ненулевых элементов строки, без плотной матрицы n_queries x n_items.
    """
    item_ids = np.asarray(item_ids)
    n_queries = query_vecs.shape[0]
    results = []

    item_vecs_T = item_vecs.T.tocsr()

    for start in range(0, n_queries, batch_size):
        end = min(start + batch_size, n_queries)
        batch_sim = query_vecs[start:end] @ item_vecs_T  # sparse (batch x n_items)
        batch_sim = batch_sim.tocsr()

        for row_idx in range(batch_sim.shape[0]):
            row_start, row_end = batch_sim.indptr[row_idx], batch_sim.indptr[row_idx + 1]
            cols = batch_sim.indices[row_start:row_end]
            vals = batch_sim.data[row_start:row_end]

            if len(cols) == 0:
                results.append([])  # ни одного пересечения по словам вообще
                continue

            if len(cols) > k:
                top_local = np.argpartition(vals, -k)[-k:]
            else:
                top_local = np.arange(len(cols))

            top_cols = cols[top_local]
            top_vals = vals[top_local]
            order = np.argsort(-top_vals)  # сортируем по убыванию скора
            results.append(item_ids[top_cols[order]].tolist())

    return results

## Отклонённый эксперимент: char n-gram TF-IDF

Для начала пробовала word-level TF-IDF, только char n-gram TF-IDF (TfidfVectorizer
и HashingVectorizer, analyzer="char_wb") и гибрид word+char как способ
компенсировать словоформы русского языка без лемматизации.

Результат: char n-gram оказался тупиковой веткой:
- **Медленнее** — нарезка текста на подстроки (3-5 символов на каждый
  символ текста) реализована в чистом Python и на 344 825 длинных
  документах занимала 500-900 секунд только на векторизацию.
- **Хуже по качеству** — recall@50 = 0.1376 против 0.1485 у word-level
  baseline. Причина: у char n-грамм нет эффективного способа отсечь
  частые "шаблонные" подстроки (в отличие от `max_df` у word-уровня),
  и короткие частые n-граммы резко снижают разреженность матрицы,
  что дополнительно замедляет и "размывает" похожесть между
  несвязанными объявлениями.

Вместо этого проблему словоформ решила стеммингом (SnowballStemmer) —
на порядок дешевле вычислительно и точнее по recall (см. ниже).

## 9. Пересборка подхода: word-level TF-IDF + стемминг + max_df

Char n-grams оказались тупиковой веткой: медленнее (нарезка длинных текстов
на подстроки — дорогая Python-операция) и хуже по качеству (нет способа
отсечь частые "шаблонные" n-граммы, как в infm_params_text).

Реализуем word-level TF-IDF, но лечим настоящую причину слабого
recall — словоформы русского языка — стеммингом (SnowballStemmer),
а не char n-граммами. Стемминг работает на уровне отдельных слов —
это на порядок дешевле, чем нарезка на все подстроки.

In [84]:

import nltk
nltk.download("punkt", quiet=True)

import re
from nltk.stem.snowball import SnowballStemmer
from tqdm.auto import tqdm

stemmer = SnowballStemmer("russian")
token_re = re.compile(r"[а-яё]+|[a-z]+|\d+")

def stem_text(text, max_words=120):
    """Токенизируем простым regex (без спец-библиотек — быстрее) и стеммим
    каждое слово. max_words ограничивает длинные описания — экономим время
    и не даём длинному описанию "перевесить" короткий, но точный title."""
    tokens = token_re.findall(text.lower())[:max_words]
    return " ".join(stemmer.stem(t) for t in tokens)

tqdm.pandas()

def build_item_text_stemmed(df, desc_words=80, infm_words=40):
    title = df["item_title_raw"].fillna("")
    infm = df["item_infm_params_text"].fillna("")
    desc = df["item_description_raw"].fillna("")

    title_stemmed = title.progress_apply(lambda t: stem_text(t, max_words=30))
    infm_stemmed = infm.progress_apply(lambda t: stem_text(t, max_words=infm_words))
    desc_stemmed = desc.progress_apply(lambda t: stem_text(t, max_words=desc_words))

    # title дублируем -- он самый точный сигнал
    return title_stemmed + " " + title_stemmed + " " + infm_stemmed + " " + desc_stemmed

def build_query_text_stemmed(df):
    q = df["search_query"].fillna("")
    infm = df["search_infm_params_text"].fillna("")
    q_stemmed = q.progress_apply(lambda t: stem_text(t, max_words=15))
    infm_stemmed = infm.progress_apply(lambda t: stem_text(t, max_words=30))
    return q_stemmed + " " + q_stemmed + " " + infm_stemmed

[nltk_data] Error loading punkt: Security Violation [pathsec.urlopen]:
[nltk_data]     refusing a proxied fetch of
[nltk_data]     'https://raw.githubusercontent.com/nltk/nltk_data/gh-
[nltk_data]     pages/index.xml'. A configured proxy performs the
[nltk_data]     egress, so NLTK cannot pin the validated IP and SSRF
[nltk_data]     protection cannot be enforced (CWE-918). If and only
[nltk_data]     if the proxy is trusted to be SSRF-safe, opt in via
[nltk_data]     NLTK_ALLOW_PROXIED_URLOPEN=1 or
[nltk_data]     nltk.pathsec.ALLOW_PROXIED_FETCH=True.


In [85]:
items_index_text_stem = build_item_text_stemmed(items_index)
val_queries_text_stem = build_query_text_stemmed(val_queries)

100%|██████████| 17724/17724 [00:01<00:00, 15826.26it/s]


In [ ]:
vectorizer_stem = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.3,      # НОВОЕ: отсекаю слова, встречающиеся более чем в 30% документов —
                      # это и есть тот самый boilerplate ("услуга", "стоимость" и т.п.)
    max_features=150_000,
    dtype=np.float32,
)
vectorizer_stem = timed("fit", vectorizer_stem.fit, items_index_text_stem)
item_vecs_stem = timed("transform items", vectorizer_stem.transform, items_index_text_stem)
val_query_vecs_stem = timed("transform val queries", vectorizer_stem.transform, val_queries_text_stem)

val_predictions_stem = timed(
    "top_k_candidates",
    top_k_candidates,
    val_query_vecs_stem, item_vecs_stem, items_index["item_id"].values, 50
)

recall_stem = recall_at_k(val_predictions_stem, val_queries["relevant_item_ids"].tolist())
print(f"Word TF-IDF + stemming + max_df recall@50: {recall_stem:.4f}")

fit: 57.1 сек
transform items: 40.9 сек
transform val queries: 0.2 сек
top_k_candidates: 22.2 сек
Word TF-IDF + stemming + max_df recall@50: 0.1797


In [87]:
# Быстрая проверка: у объявления, которое реально выбрали по запросу,
# category_id почти всегда совпадает с search_category?
check = train[["search_category", "item_category_id"]].copy()
match_rate = (check["search_category"] == check["item_category_id"]).mean()
print(f"Доля пар, где search_category == item_category_id: {match_rate:.4f}")

# То же самое для location_id (тут ожидаю более слабое совпадение —
# это конкретный адрес/район, а не просто "тот же город")
loc_match_rate = (train["search_location_id"] == train["item_location_id"]).mean()
print(f"Доля пар, где search_location_id == item_location_id: {loc_match_rate:.4f}")

Доля пар, где search_category == item_category_id: 0.9999
Доля пар, где search_location_id == item_location_id: 0.8310


## 10. Добавляю буст за совпадение категории/локации к TF-IDF скору

Модифицирую top_k_candidates: считаю TF-IDF cosine similarity как раньше,
но добавляю фиксированный бонус к скору тех объявлений, у которых
item_category_id == search_category (и/или item_location_id == search_location_id).
Бонус подбираю как гиперпараметр (сравним 0 / 0.05 / 0.1 / 0.2 по recall на валидации).

In [88]:
def top_k_candidates_boosted(
    query_vecs, item_vecs, item_ids,
    query_categories, item_categories,
    query_locations=None, item_locations=None,
    category_boost=0.1, location_boost=0.0,
    k=50, batch_size=500,
):
    """Тот же top_k_candidates, но с бустом за совпадение категории/локации.
    Буст добавляется как константа к скору (а не умножение), чтобы не обнулять
    объявления с category mismatch, но нулевым текстовым сходством с релевантным."""
    item_ids = np.asarray(item_ids)
    item_categories = np.asarray(item_categories)
    if item_locations is not None:
        item_locations = np.asarray(item_locations)

    n_queries = query_vecs.shape[0]
    results = []
    item_vecs_T = item_vecs.T.tocsr()

    for start in range(0, n_queries, batch_size):
        end = min(start + batch_size, n_queries)
        batch_sim = (query_vecs[start:end] @ item_vecs_T).tocsr()

        for local_idx, row_idx in enumerate(range(start, end)):
            row_start, row_end = batch_sim.indptr[local_idx], batch_sim.indptr[local_idx + 1]
            cols = batch_sim.indices[row_start:row_end]
            vals = batch_sim.data[row_start:row_end].copy()

            if len(cols) == 0:
                results.append([])
                continue

            # буст за категорию
            cat_match = item_categories[cols] == query_categories[row_idx]
            vals = vals + cat_match * category_boost

            # буст за локацию (опционально)
            if location_boost > 0 and item_locations is not None:
                loc_match = item_locations[cols] == query_locations[row_idx]
                vals = vals + loc_match * location_boost

            if len(cols) > k:
                top_local = np.argpartition(vals, -k)[-k:]
            else:
                top_local = np.arange(len(cols))

            top_cols = cols[top_local]
            top_vals = vals[top_local]
            order = np.argsort(-top_vals)
            results.append(item_ids[top_cols[order]].tolist())

    return results

In [89]:
query_categories_val = val_queries["search_category"].values
item_categories_idx = items_index["item_category_id"].values

for boost in [0.0, 0.05, 0.1, 0.2, 0.3]:
    preds = top_k_candidates_boosted(
        val_query_vecs_stem, item_vecs_stem, items_index["item_id"].values,
        query_categories_val, item_categories_idx,
        category_boost=boost, k=50,
    )
    r = recall_at_k(preds, val_queries["relevant_item_ids"].tolist())
    print(f"category_boost={boost}: recall@50 = {r:.4f}")

category_boost=0.0: recall@50 = 0.1797
category_boost=0.05: recall@50 = 0.1797
category_boost=0.1: recall@50 = 0.1797
category_boost=0.2: recall@50 = 0.1797
category_boost=0.3: recall@50 = 0.1797


In [90]:
print("Уникальных значений item_category_id в items_index:", items_index["item_category_id"].nunique())
print("Уникальных значений item_microcat_id в items_index:", items_index["item_microcat_id"].nunique())
print("Уникальных значений search_category в train_queries:", train_queries["search_category"].nunique())

print("\nТоп-10 значений item_category_id по частоте:")
print(items_index["item_category_id"].value_counts().head(10))

# Ключевая проверка: для каждой строки items_index — какая доля ВСЕХ объявлений
# в индексе имеет ту же category_id, что и типичный запрос (например, 114)?
print("\nДоля объявлений с category_id == 114 (значение из твоих примеров):")
print((items_index["item_category_id"] == 114).mean())

Уникальных значений item_category_id в items_index: 8
Уникальных значений item_microcat_id в items_index: 212
Уникальных значений search_category в train_queries: 4

Топ-10 значений item_category_id по частоте:
item_category_id
114    344814
40          2
27          2
19          2
112         2
81          1
28          1
10          1
Name: count, dtype: int64

Доля объявлений с category_id == 114 (значение из твоих примеров):
0.9999680997607482


## 11. Буст по item_microcat_id через "типичную микрокатегорию запроса"

item_category_id вырожден (99.99% = одно значение) — бесполезен.
item_microcat_id осмысленный (212 значений), но у search-признаков нет
прямого аналога. Строю query -> most common microcat по частоте выбора
в train (только по train_queries, чтобы не было утечки на val), и
использую это как дополнительный буст.

In [ ]:
# Разворачиваю train обратно к строкам пар (нам нужен train ИСХОДНЫЙ,
# отфильтрованный только по запросам, попавшим в train_queries -- чтобы
# избежать утечки данных про val_queries).
train_query_keys = set(map(tuple, train_queries[search_cols].astype(str).values))

train_pairs_for_microcat = train.copy()
train_pairs_for_microcat["_key"] = list(
    map(tuple, train_pairs_for_microcat[search_cols].astype(str).values)
)
train_pairs_for_microcat = train_pairs_for_microcat[
    train_pairs_for_microcat["_key"].isin(train_query_keys)
]

# Для простоты и обобщаемости на бенчмарк использую как ключ ТОЛЬКО
# search_query (текст) -- т.к. именно текст запроса будет пересекаться между
# train и бенчмарком, а полный набор search_*-полей вряд ли повторится 1-в-1.
query_text_to_microcat = (
    train_pairs_for_microcat.groupby("search_query")["item_microcat_id"]
    .agg(lambda s: s.value_counts().idxmax())  # самая частая микрокатегория для этого текста запроса
    .to_dict()
)
print("Построено сопоставлений query_text -> microcat:", len(query_text_to_microcat))

# Покрытие: для какой доли val-запросов у нас есть предсказанная microcat?
val_has_microcat = val_queries["search_query"].isin(query_text_to_microcat).mean()
print(f"Доля val-запросов с известной microcat (по совпадению текста с train): {val_has_microcat:.4f}")

Построено сопоставлений query_text -> microcat: 72232
Доля val-запросов с известной microcat (по совпадению текста с train): 0.8689


In [92]:
val_queries_microcat = val_queries["search_query"].map(query_text_to_microcat).fillna(-1).values
item_microcat_idx = items_index["item_microcat_id"].values

for boost in [0.0, 0.1, 0.2, 0.3, 0.5]:
    preds = top_k_candidates_boosted(
        val_query_vecs_stem, item_vecs_stem, items_index["item_id"].values,
        val_queries_microcat, item_microcat_idx,
        category_boost=boost, k=50,
    )
    r = recall_at_k(preds, val_queries["relevant_item_ids"].tolist())
    print(f"microcat_boost={boost}: recall@50 = {r:.4f}")

microcat_boost=0.0: recall@50 = 0.1797
microcat_boost=0.1: recall@50 = 0.1837
microcat_boost=0.2: recall@50 = 0.1820
microcat_boost=0.3: recall@50 = 0.1775
microcat_boost=0.5: recall@50 = 0.1716


In [93]:
val_query_locations = val_queries["search_location_id"].values
item_locations_idx = items_index["item_location_id"].values

results = []
for m_boost in [0.0, 0.1, 0.15]:
    for l_boost in [0.0, 0.05, 0.1, 0.15]:
        preds = top_k_candidates_boosted(
            val_query_vecs_stem, item_vecs_stem, items_index["item_id"].values,
            val_queries_microcat, item_microcat_idx,
            query_locations=val_query_locations, item_locations=item_locations_idx,
            category_boost=m_boost, location_boost=l_boost, k=50,
        )
        r = recall_at_k(preds, val_queries["relevant_item_ids"].tolist())
        results.append((m_boost, l_boost, r))
        print(f"microcat_boost={m_boost}, location_boost={l_boost}: recall@50 = {r:.4f}")

best = max(results, key=lambda x: x[2])
print(f"\nЛучшая комбинация: microcat_boost={best[0]}, location_boost={best[1]}, recall@50={best[2]:.4f}")

microcat_boost=0.0, location_boost=0.0: recall@50 = 0.1797
microcat_boost=0.0, location_boost=0.05: recall@50 = 0.2404
microcat_boost=0.0, location_boost=0.1: recall@50 = 0.3296
microcat_boost=0.0, location_boost=0.15: recall@50 = 0.4299
microcat_boost=0.1, location_boost=0.0: recall@50 = 0.1837
microcat_boost=0.1, location_boost=0.05: recall@50 = 0.2453
microcat_boost=0.1, location_boost=0.1: recall@50 = 0.3339
microcat_boost=0.1, location_boost=0.15: recall@50 = 0.4360
microcat_boost=0.15, location_boost=0.0: recall@50 = 0.1825
microcat_boost=0.15, location_boost=0.05: recall@50 = 0.2442
microcat_boost=0.15, location_boost=0.1: recall@50 = 0.3316
microcat_boost=0.15, location_boost=0.15: recall@50 = 0.4291

Лучшая комбинация: microcat_boost=0.1, location_boost=0.15, recall@50=0.4360


In [94]:
print("Уникальных location_id в items_index:", items_index["item_location_id"].nunique())
print("Уникальных search_location_id в val_queries:", val_queries["search_location_id"].nunique())

# сколько объявлений в среднем приходится на один location_id -- 
# если мало (десятки-сотни), это подтверждает гипотезу про гранулярность
loc_sizes = items_index["item_location_id"].value_counts()
print("\nРаспределение размера location-групп (сколько объявлений на один location_id):")
print(loc_sizes.describe())

Уникальных location_id в items_index: 2637
Уникальных search_location_id в val_queries: 1172

Распределение размера location-групп (сколько объявлений на один location_id):
count     2637.000000
mean       130.764126
std        795.252990
min          1.000000
25%          2.000000
50%          6.000000
75%         33.000000
max      26663.000000
Name: count, dtype: float64


In [95]:
results2 = []
for l_boost in [0.15, 0.2, 0.3, 0.5, 1.0, 2.0]:
    preds = top_k_candidates_boosted(
        val_query_vecs_stem, item_vecs_stem, items_index["item_id"].values,
        val_queries_microcat, item_microcat_idx,
        query_locations=val_query_locations, item_locations=item_locations_idx,
        category_boost=0.1, location_boost=l_boost, k=50,
    )
    r = recall_at_k(preds, val_queries["relevant_item_ids"].tolist())
    results2.append((l_boost, r))
    print(f"location_boost={l_boost}: recall@50 = {r:.4f}")

location_boost=0.15: recall@50 = 0.4360
location_boost=0.2: recall@50 = 0.5315
location_boost=0.3: recall@50 = 0.6682
location_boost=0.5: recall@50 = 0.7563
location_boost=1.0: recall@50 = 0.7632
location_boost=2.0: recall@50 = 0.7632


In [96]:
for m_boost in [0.0, 0.05, 0.1, 0.15, 0.2]:
    preds = top_k_candidates_boosted(
        val_query_vecs_stem, item_vecs_stem, items_index["item_id"].values,
        val_queries_microcat, item_microcat_idx,
        query_locations=val_query_locations, item_locations=item_locations_idx,
        category_boost=m_boost, location_boost=1.0, k=50,
    )
    r = recall_at_k(preds, val_queries["relevant_item_ids"].tolist())
    print(f"microcat_boost={m_boost}, location_boost=1.0: recall@50 = {r:.4f}")

microcat_boost=0.0, location_boost=1.0: recall@50 = 0.7463
microcat_boost=0.05, location_boost=1.0: recall@50 = 0.7629
microcat_boost=0.1, location_boost=1.0: recall@50 = 0.7632
microcat_boost=0.15, location_boost=1.0: recall@50 = 0.7620
microcat_boost=0.2, location_boost=1.0: recall@50 = 0.7601


In [97]:
# Для каждого val-запроса: есть ли релевантный item среди объявлений С ТОЙ ЖЕ локацией?
items_by_loc = items_index.groupby("item_location_id")["item_id"].apply(set).to_dict()

def relevant_in_same_location(row):
    loc = row["search_location_id"]
    rel = set(row["relevant_item_ids"])
    same_loc_items = items_by_loc.get(loc, set())
    return len(rel & same_loc_items) > 0

val_queries["_has_loc_match"] = val_queries.apply(relevant_in_same_location, axis=1)
print("Доля val-запросов, где релевантный item ЕСТЬ в той же локации:", val_queries["_has_loc_match"].mean())

# Из тех, где локация в принципе совпадает -- какой размер location-группы
# (если группа большая, там и TF-IDF-часть должна отработать, чтобы не потерять в top-50)
matched = val_queries[val_queries["_has_loc_match"]]
group_sizes = matched["search_location_id"].map(lambda l: len(items_by_loc.get(l, set())))
print("\nРазмер location-группы для запросов, где релевантный item в той же локации:")
print(group_sizes.describe())

Доля val-запросов, где релевантный item ЕСТЬ в той же локации: 0.8050101557210562

Размер location-группы для запросов, где релевантный item в той же локации:
count    14268.000000
mean      4576.485212
std       6941.039516
min          1.000000
25%        596.000000
50%       2055.000000
75%       4787.000000
max      26663.000000
Name: search_location_id, dtype: float64


In [98]:
# Гипотеза: те 19.5% запросов, где релевантный item НЕ в той же локации,
# чаще связаны с доставкой/выездом -- тогда точная локация менее важна.
print("Среди запросов БЕЗ location-match:")
no_match = val_queries[~val_queries["_has_loc_match"]]
print(no_match["search_is_delivery_search"].value_counts(normalize=True))

print("\nСреди запросов С location-match:")
match = val_queries[val_queries["_has_loc_match"]]
print(match["search_is_delivery_search"].value_counts(normalize=True))

Среди запросов БЕЗ location-match:
search_is_delivery_search
0    1.0
Name: proportion, dtype: float64

Среди запросов С location-match:
search_is_delivery_search
0    1.0
Name: proportion, dtype: float64


## 13. Качественный анализ ошибок: смотрим на конкретные провалы

Посмотрим на реальные случаи, где релевантный item есть в той же локации, но не попал в top-50.
Гипотеза: возможно, у части таких пар просто нет пересечения слов между
запросом и объявлением (синонимы/другая формулировка) — тогда это
ограничение самого TF-IDF, а не проблема весов.

In [99]:
# Пересчитаем предсказания на лучшей текущей комбинации весов
best_preds = top_k_candidates_boosted(
    val_query_vecs_stem, item_vecs_stem, items_index["item_id"].values,
    val_queries_microcat, item_microcat_idx,
    query_locations=val_query_locations, item_locations=item_locations_idx,
    category_boost=0.1, location_boost=1.0, k=50,
)
val_queries["_pred"] = best_preds

def is_failed_but_loc_matched(row):
    if not row["_has_loc_match"]:
        return False
    rel = set(row["relevant_item_ids"])
    pred = set(row["_pred"])
    return len(rel & pred) == 0  # ни одного релевантного не поймали, хотя локация совпадает

val_queries["_failed_loc_matched"] = val_queries.apply(is_failed_but_loc_matched, axis=1)
failed = val_queries[val_queries["_failed_loc_matched"]]
print("Таких провалов:", len(failed))

items_by_id = items_index.set_index("item_id")

# печатаем 10 случайных провалов: запрос vs заголовок реально релевантного item
sample = failed.sample(10, random_state=42)
for _, row in sample.iterrows():
    rel_id = row["relevant_item_ids"][0]
    if rel_id in items_by_id.index:
        title = items_by_id.loc[rel_id, "item_title_raw"]
        print(f"ЗАПРОС: {row['search_query']!r}  ->  РЕЛЕВАНТНЫЙ TITLE: {title!r}")
    else:
        print(f"ЗАПРОС: {row['search_query']!r}  ->  релевантный item не найден в items_index (?)")

Таких провалов: 1263
ЗАПРОС: 'перевозки на дачу'  ->  РЕЛЕВАНТНЫЙ TITLE: 'Грузоперевозки, доставка по спб и Лен.области'
ЗАПРОС: 'ремонт бытовой техники'  ->  РЕЛЕВАНТНЫЙ TITLE: 'Ремонт электроплит'
ЗАПРОС: 'вымыть окна в квартире'  ->  РЕЛЕВАНТНЫЙ TITLE: 'Сезонная мойка окон. Клининг. Уборка'
ЗАПРОС: 'стрижка кончиков'  ->  РЕЛЕВАНТНЫЙ TITLE: 'Кератиновое выпрямление'
ЗАПРОС: 'модель милирование'  ->  РЕЛЕВАНТНЫЙ TITLE: 'Парикмахерские услуги/окрашивание/стрижки'
ЗАПРОС: 'похудение'  ->  РЕЛЕВАНТНЫЙ TITLE: 'Услуги манипулятора'
ЗАПРОС: 'автоподборщик автоподбор'  ->  РЕЛЕВАНТНЫЙ TITLE: 'Автоэксперт / Разовый осмотр авто / Эндоскопия'
ЗАПРОС: 'пошив худи'  ->  РЕЛЕВАНТНЫЙ TITLE: 'Ателье по пошиву одежды'
ЗАПРОС: 'разравнять землю'  ->  РЕЛЕВАНТНЫЙ TITLE: 'Поднятие и отсыпка участка'
ЗАПРОС: 'снять домик глэмпинг с чаном'  ->  РЕЛЕВАНТНЫЙ TITLE: 'Ужин на берегу моря'


## 14. Быстрые улучшения текстовой части: sublinear_tf + rating tie-break

Два дешёвых, но потенциально полезных изменения:
1. sublinear_tf=True — заменяет частоту слова tf на 1+log(tf) при расчёте
   TF-IDF. Смысл: слово, встретившееся в тексте 10 раз, не должно весить
   в 10 раз больше слова, встретившегося 1 раз — разница в 1-2 упоминания
   значит намного меньше, чем линейный рост. Особенно полезно для длинных
   description, где ключевое слово может случайно повториться много раз.
2. Rating/reviews tie-break — внутри одной location-группы (где могут
   конкурировать тысячи объявлений с похожим текстом) добавляю небольшой
   бонус более "проверенным" исполнителям: это не замена текстового
   сходства, а лёгкий сдвиг при примерно равном текстовом скоре.

In [100]:
vectorizer_stem2 = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.3,
    max_features=150_000,
    sublinear_tf=True,   # НОВОЕ
    dtype=np.float32,
)
vectorizer_stem2 = timed("fit", vectorizer_stem2.fit, items_index_text_stem)
item_vecs_stem2 = timed("transform items", vectorizer_stem2.transform, items_index_text_stem)
val_query_vecs_stem2 = timed("transform val queries", vectorizer_stem2.transform, val_queries_text_stem)

preds_sublinear = top_k_candidates_boosted(
    val_query_vecs_stem2, item_vecs_stem2, items_index["item_id"].values,
    val_queries_microcat, item_microcat_idx,
    query_locations=val_query_locations, item_locations=item_locations_idx,
    category_boost=0.1, location_boost=1.0, k=50,
)
recall_sublinear = recall_at_k(preds_sublinear, val_queries["relevant_item_ids"].tolist())
print(f"sublinear_tf=True (+ те же бусты) recall@50: {recall_sublinear:.4f}  (было 0.7632)")

fit: 59.0 сек
transform items: 40.8 сек
transform val queries: 0.2 сек
sublinear_tf=True (+ те же бусты) recall@50: 0.7661  (было 0.7632)


In [101]:
def top_k_candidates_boosted_v2(
    query_vecs, item_vecs, item_ids,
    query_categories, item_categories,
    query_locations, item_locations,
    item_rating=None, item_reviews=None,
    category_boost=0.1, location_boost=1.0, rating_boost=0.0,
    k=50, batch_size=500,
):
    """Добавляем rating_boost: небольшой бонус, пропорциональный
    rating * log1p(reviews_count) -- нормализуем заранее в [0, 1],
    чтобы rating_boost был сопоставим по масштабу с category/location."""
    item_ids = np.asarray(item_ids)
    item_categories = np.asarray(item_categories)
    item_locations = np.asarray(item_locations)

    if item_rating is not None:
        rating_score = np.nan_to_num(np.asarray(item_rating), nan=0.0) / 5.0  # rating 0-5 -> 0-1
        reviews_score = np.log1p(np.nan_to_num(np.asarray(item_reviews), nan=0.0))
        reviews_score = reviews_score / (reviews_score.max() + 1e-9)  # 0-1
        quality_score = rating_score * reviews_score  # оба сигнала должны быть высокими одновременно
    else:
        quality_score = np.zeros(len(item_ids))

    n_queries = query_vecs.shape[0]
    results = []
    item_vecs_T = item_vecs.T.tocsr()

    for start in range(0, n_queries, batch_size):
        end = min(start + batch_size, n_queries)
        batch_sim = (query_vecs[start:end] @ item_vecs_T).tocsr()

        for local_idx, row_idx in enumerate(range(start, end)):
            row_start, row_end = batch_sim.indptr[local_idx], batch_sim.indptr[local_idx + 1]
            cols = batch_sim.indices[row_start:row_end]
            vals = batch_sim.data[row_start:row_end].copy()

            if len(cols) == 0:
                results.append([])
                continue

            cat_match = item_categories[cols] == query_categories[row_idx]
            vals = vals + cat_match * category_boost

            loc_match = item_locations[cols] == query_locations[row_idx]
            vals = vals + loc_match * location_boost

            vals = vals + quality_score[cols] * rating_boost

            if len(cols) > k:
                top_local = np.argpartition(vals, -k)[-k:]
            else:
                top_local = np.arange(len(cols))

            top_cols = cols[top_local]
            top_vals = vals[top_local]
            order = np.argsort(-top_vals)
            results.append(item_ids[top_cols[order]].tolist())

    return results

item_rating_idx = items_index["item_rating"].values
item_reviews_idx = items_index["item_rating_reviews_count"].values

for r_boost in [0.0, 0.02, 0.05, 0.1]:
    preds = top_k_candidates_boosted_v2(
        val_query_vecs_stem2, item_vecs_stem2, items_index["item_id"].values,
        val_queries_microcat, item_microcat_idx,
        val_query_locations, item_locations_idx,
        item_rating=item_rating_idx, item_reviews=item_reviews_idx,
        category_boost=0.1, location_boost=1.0, rating_boost=r_boost, k=50,
    )
    r = recall_at_k(preds, val_queries["relevant_item_ids"].tolist())
    print(f"rating_boost={r_boost}: recall@50 = {r:.4f}")

rating_boost=0.0: recall@50 = 0.7661
rating_boost=0.02: recall@50 = 0.7657
rating_boost=0.05: recall@50 = 0.7656
rating_boost=0.1: recall@50 = 0.7620


## 17. Финальный инференс на benchmark_queries / benchmark_items

Гиперпараметры зафиксированы по результатам валидации:
- TF-IDF: word-level, ngram_range=(1,2), min_df=3, max_df=0.3,
  max_features=150_000, sublinear_tf=True, стемминг (SnowballStemmer ru)
- Бусты: microcat_boost=0.1 (через query_text -> most common microcat
  по всему train), location_boost=1.0
- Финальный recall@50 на валидации: 0.7661

Дальше train/val-разбиение не используется -- строим финальную модель
на всём train + ищем среди benchmark_items.

In [102]:
bench_items_text_stem = build_item_text_stemmed(bench_items)
bench_queries_text_stem = build_query_text_stemmed(bench_queries)

print("Пример готового текста объявления:", bench_items_text_stem.iloc[0][:200])
print("Пример готового текста запроса:", bench_queries_text_stem.iloc[0][:200])

100%|██████████| 2452/2452 [00:00<00:00, 30651.13it/s]

Пример готового текста объявления: ремонт выкуп компьют и ноутбук с выезд на дом ремонт выкуп компьют и ноутбук с выезд на дом вид услуг компьютерн помощ мест оказан услуг пр т ленин тип стоимост за услуг начальн цен график работ от 34
Пример готового текста запроса: перевозк владикавказ тбилис перевозк владикавказ тбилис 


In [103]:
final_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.3,
    max_features=150_000,
    sublinear_tf=True,
    dtype=np.float32,
)
final_vectorizer = timed("fit (на benchmark_items)", final_vectorizer.fit, bench_items_text_stem)
final_item_vecs = timed("transform bench_items", final_vectorizer.transform, bench_items_text_stem)
final_query_vecs = timed("transform bench_queries", final_vectorizer.transform, bench_queries_text_stem)

fit (на benchmark_items): 32.3 сек
transform bench_items: 23.0 сек
transform bench_queries: 0.0 сек


In [ ]:
# Теперь использую ВЕСЬ train (не только train_queries, как на этапе валидации) --
# больше данных для этого lookup'а, утечки нет, т.к. это не текстовый признак
# объявления, а отдельная эвристика поверх сырых train-логов.
full_query_text_to_microcat = (
    train.groupby("search_query")["item_microcat_id"]
    .agg(lambda s: s.value_counts().idxmax())
    .to_dict()
)
print("Словарь query -> microcat (весь train):", len(full_query_text_to_microcat))

bench_queries_microcat = bench_queries["search_query"].map(full_query_text_to_microcat).fillna(-1).values
coverage = (bench_queries_microcat != -1).mean()
print(f"Покрытие microcat для benchmark_queries: {coverage:.4f}")

Словарь query -> microcat (весь train): 74529
Покрытие microcat для benchmark_queries: 0.3699


In [ ]:
bench_item_categories = bench_items["item_microcat_id"].values  # переиспользую как "microcat" сигнал
bench_query_locations = bench_queries["search_location_id"].values
bench_item_locations = bench_items["item_location_id"].values

final_predictions = top_k_candidates_boosted(
    final_query_vecs, final_item_vecs, bench_items["item_id"].values,
    bench_queries_microcat, bench_item_categories,
    query_locations=bench_query_locations, item_locations=bench_item_locations,
    category_boost=0.1, location_boost=1.0, k=50,
)

print("Готово предсказаний:", len(final_predictions))
print("Пример top-50 для первого запроса:", final_predictions[0][:5], "...")

# Sanity check: у всех ли запросов есть хоть какие-то кандидаты?
empty_count = sum(1 for p in final_predictions if len(p) == 0)
print(f"Запросов с пустым списком кандидатов: {empty_count}")

Готово предсказаний: 2452
Пример top-50 для первого запроса: ['d722bcda1a555091', '603623b4bd9e8f6c', '355392014208b7bf', 'e08f4ce94c268cef', '6042fecdd7753112'] ...
Запросов с пустым списком кандидатов: 9


## 18. Обработка edge case: пустые предсказания

Если у какого-то запроса TF-IDF не нашёл ни одного пересечения слов
(редкий случай - специфичные опечатки, очень короткие/нетипичные запросы),
top_k_candidates вернёт пустой список - 0 из 50 мест использовано,
это прямая потеря recall без необходимости. Нужен fallback.

In [ ]:
# Fallback: для запросов без TF-IDF-пересечений берём топ-50 объявлений
# из той же локации (без учёта текста) - лучше, чем пустой список.
items_by_loc_bench = bench_items.groupby("item_location_id")["item_id"].apply(list).to_dict()

def apply_fallback(predictions, query_locations, items_by_loc, k=50):
    fixed = []
    n_used_fallback = 0
    for pred, loc in zip(predictions, query_locations):
        if len(pred) == 0:
            fallback_items = items_by_loc.get(loc, [])[:k]
            fixed.append(fallback_items)
            n_used_fallback += 1
        else:
            fixed.append(pred)
    print(f"Fallback применён к {n_used_fallback} запросам")
    return fixed

final_predictions_fixed = apply_fallback(final_predictions, bench_query_locations, items_by_loc_bench, k=50)

# Финальная проверка: нет ли всё ещё пустых списков (например, если и в
# локации вообще нет объявлений - тогда честно оставляем пустым, лучше
# отдать хоть что-то другое, чем пусто)
still_empty = sum(1 for p in final_predictions_fixed if len(p) == 0)
print("Всё ещё пусто после fallback:", still_empty)

Fallback применён к 9 запросам
Всё ещё пусто после fallback: 1


In [ ]:
# Найдём, что за запрос остался с пустым списком - полезно для отчёта
empty_idx = [i for i, p in enumerate(final_predictions_fixed) if len(p) == 0]
print("Индексы всё ещё пустых запросов:", empty_idx)
for i in empty_idx:
    print(bench_queries.iloc[i][["query_id", "search_query", "search_location_id"]])

# Второй уровень fallback: топ-50 объявлений по рейтингу*отзывам ГЛОБАЛЬНО
# (без привязки к локации/тексту вообще) - это "последний резерв", чтобы
# гарантированно не оставить ни одного query_id без ответа.
bench_items_quality = (
    bench_items["item_rating"].fillna(0) * np.log1p(bench_items["item_rating_reviews_count"].fillna(0))
)
global_top_fallback = (
    bench_items.assign(_q=bench_items_quality.values)
    .sort_values("_q", ascending=False)
    .head(50)["item_id"]
    .tolist()
)

final_predictions_final = [
    global_top_fallback if len(pred) == 0 else pred
    for pred in final_predictions_fixed
]

still_empty_final = sum(1 for p in final_predictions_final if len(p) == 0)
print("\nВсё ещё пусто после второго fallback:", still_empty_final)
print("Все длины <= 50:", all(len(p) <= 50 for p in final_predictions_final))

Индексы всё ещё пустых запросов: [20]
query_id              Mkw7pexvDmMpFdzO
search_query                 натуропат
search_location_id              657310
Name: 20, dtype: object

Всё ещё пусто после второго fallback: 0
Все длины <= 50: True


## 19. Сборка answer.csv и валидация формата

Проверяю по всем требованиям задания:
- ровно одна строка на каждый query_id из benchmark_queries, без пропусков/повторов
- не больше 50 item_id в строке, без дублей внутри строки
- все item_id существуют в benchmark_items
- item_id и query_id - строки, а не числа (не теряем ведущие нули,
  не улетаем в экспоненциальную запись)
- ровно две колонки, без индекса

In [ ]:
answer = pd.DataFrame({
    "query_id": bench_queries["query_id"].astype(str).values,
    "answer": [" ".join(preds) for preds in final_predictions_final],
})

print(answer.shape)
answer.head(3)

(2452, 2)


,query_id,answer
0,70DfDUpwjxB4lzFd,d722bcda1a555091 603623b4bd9e8f6c 355392014208b7bf e08f4ce94c268cef 6042fecdd7753112 936c0eb79b64074d b92ee8f432cec2...
1,JTrdTaZJvSiLPkXj,422d3ffdd5bbf626 8703b2a0442c5d25 cadb904468479c51 16ea26ff0aa0450a 413f6384e3c07b64 4a435cc0a254c1fd 431efbc80d5915...
2,LZCZNoVG4AFUkVRJ,d8fce513e4f000a7 3f89b8062dc85f1c 168a9207e80b0be4 dab52187b4500d9b edecb3695ab88901 d62074dd39caefee 5c3ee000a94c97...


In [ ]:
errors = []

# 1. Ровно одна строка на каждый query_id, без пропусков и повторов
expected_qids = set(bench_queries["query_id"].astype(str))
actual_qids = set(answer["query_id"])
if expected_qids != actual_qids:
    errors.append(f"Несовпадение query_id: пропущено {len(expected_qids - actual_qids)}, лишних {len(actual_qids - expected_qids)}")
if answer["query_id"].duplicated().any():
    errors.append(f"Дубли query_id: {answer['query_id'].duplicated().sum()}")

# 2. Формат item_id: ровно 16 символов, 0-9a-f
valid_item_ids = set(bench_items["item_id"].astype(str))
hex16_re = re.compile(r"^[0-9a-f]{16}$")

max_len_seen = 0
for _, row in answer.iterrows():
    ids = row["answer"].split(" ") if row["answer"] else []
    max_len_seen = max(max_len_seen, len(ids))

    if len(ids) > 50:
        errors.append(f"{row['query_id']}: больше 50 id ({len(ids)})")
    if len(ids) != len(set(ids)):
        errors.append(f"{row['query_id']}: дубли item_id внутри строки")
    for iid in ids:
        if not hex16_re.match(iid):
            errors.append(f"{row['query_id']}: некорректный формат item_id '{iid}'")
        elif iid not in valid_item_ids:
            errors.append(f"{row['query_id']}: item_id '{iid}' не найден в benchmark_items")

print("Макс. число id в одной строке:", max_len_seen)
print("Среднее число id в строке:", answer["answer"].str.split().str.len().mean())

if errors:
    print(f"\n НАЙДЕНО {len(errors)} ОШИБОК, первые 10:")
    for e in errors[:10]:
        print(" -", e)
else:
    print("\nВсе проверки пройдены, ошибок не найдено.")

Макс. число id в одной строке: 50
Среднее число id в строке: 49.701876019575856

Все проверки пройдены, ошибок не найдено.


In [ ]:
# Убедимся, что типы - строки 
assert answer["query_id"].apply(type).eq(str).all()
assert answer["answer"].apply(type).eq(str).all()
assert list(answer.columns) == ["query_id", "answer"]  # ровно эти 2 колонки, в этом порядке

answer.to_csv("answer.csv", index=False, encoding="utf-8")
print("Сохранено: answer.csv")

# Перечитываем с диска и ещё раз проверяем - защита от сюрпризов сериализации 
reread = pd.read_csv("answer.csv", dtype=str)
print("\nПосле перечитывания с диска:")
print(reread.dtypes)
print(reread.head(3))
print("Число строк:", len(reread))
assert len(reread) == len(bench_queries), "Число строк не совпадает с benchmark_queries!"
print("\n Финальная проверка пройдена")

Сохранено: answer.csv

После перечитывания с диска:
query_id    str
answer      str
dtype: object
           query_id  \
0  70DfDUpwjxB4lzFd   
1  JTrdTaZJvSiLPkXj   
2  LZCZNoVG4AFUkVRJ   

                                                                                                                    answer  
0  d722bcda1a555091 603623b4bd9e8f6c 355392014208b7bf e08f4ce94c268cef 6042fecdd7753112 936c0eb79b64074d b92ee8f432cec2...  
1  422d3ffdd5bbf626 8703b2a0442c5d25 cadb904468479c51 16ea26ff0aa0450a 413f6384e3c07b64 4a435cc0a254c1fd 431efbc80d5915...  
2  d8fce513e4f000a7 3f89b8062dc85f1c 168a9207e80b0be4 dab52187b4500d9b edecb3695ab88901 d62074dd39caefee 5c3ee000a94c97...  
Число строк: 2452

✅ Финальная проверка пройдена


## 20. Доработка: прямой парсинг тегов услуги из infm_params_text

 Вместо lookup через train попробуем извлекать
категориальные теги ("Вид услуги", "Тип услуги") напрямую из текста
регуляркой -- это работает на любом новом запросе, не завязано на train.

In [111]:
import re

# Смотрим на сырые примеры, чтобы понять точный формат разметки полей
print("--- search_infm_params_text, непустые примеры ---")
for t in train["search_infm_params_text"].dropna():
    if t.strip():
        print(repr(t))
    if train["search_infm_params_text"].dropna().tolist().index(t) > 15:
        break

--- search_infm_params_text, непустые примеры ---
'Рейтинг пользователя 4 звезды и выше'
'Онлайн-запись Тип услуги СПА-услуги, массаж Вид услуги Красота, здоровье'
'Вид услуги Оборудование, производство'
'Тип услуги Ремонт квартир и домов под ключ Вид услуги Ремонт и отделка'
'Тип услуги Сборка и ремонт мебели Вид услуги Ремонт и отделка'
'Онлайн-запись Тип услуги Маникюр, педикюр Вид услуги Красота, здоровье'
'Вид услуги'
'Где вы оказываете услуги У себя дома Ваши клиенты Мужчины Кто оказывает услуги Женщина Тип услуги СПА-услуги, массаж Вид услуги Красота, здоровье'
'Вид услуги Автосервис, аренда Тип услуги Автосервис'
'Тип услуги автосервиса Кузовной ремонт и покраска Вид услуги Автосервис, аренда Тип услуги Автосервис'
'Тип услуги Другое Вид услуги Доставка еды и продуктов'
'Тип услуги автосервиса Кузовной ремонт и покраска Вид услуги Автосервис, аренда Тип услуги Автосервис'
'Вид услуги Ремонт и отделка'
'Вид услуги Красота, здоровье'


In [112]:
import re
from collections import Counter

# Известные метки из search-стороны (видно по примерам выше) -- небольшой закрытый список
KNOWN_LABELS_BASE = [
    "Тип услуги автосервиса", "Тип стоимости за услугу", "Место оказания услуг",
    "Где вы оказываете услуги", "Куда выезжаете", "Кто оказывает услуги",
    "Ваши клиенты", "Онлайн-запись", "Рейтинг пользователя",
    "Начальная цена", "Название услуги", "Опыт работы", "Год окончания",
    "Специальность", "Учебное учреждение", "Работа по договору",
    "Гарантия на работу", "Минимальная сумма заказа", "Как вы работаете",
    "Работаете с юрлицами и ИП", "Работа с техникой премиум-класса",
    "Высотные работы", "Производители", "Тип оборудования", "Марка",
    "Модель", "Доставка", "Минимальное время аренды", "Техника",
    "Тип подъёмной техники", "Грузоподъёмность борта", "Грузоподъёмность стрелы",
    "Длина борта", "Длина стрелы", "Вездеход", "С оператором",
    "Тип стоимости за час", "Применен автозаголовок", "Районы",
    "График работы от", "График работы до", "Время работы, с",
    "Время работы, до", "Продолжительность", "Стоимость", "Услуга",
    "Дни пн", "Дни вт", "Дни ср", "Дни чт", "Дни пт", "Дни сб", "Дни вс",
    "Вид услуги", "Тип услуги",  # два целевых, оставляем последними -- они самые короткие/общие
]
# Сортируем по убыванию длины -- иначе короткое "Тип услуги" перехватит
# совпадение раньше, чем "Тип услуги автосервиса" -- классическая ловушка regex-alternation
KNOWN_LABELS = sorted(set(KNOWN_LABELS_BASE), key=len, reverse=True)

label_pattern = re.compile("|".join(re.escape(l) for l in KNOWN_LABELS))

def extract_tags(text, target_labels=("Вид услуги", "Тип услуги")):
    """Разбиваем текст на сегменты по известным меткам, для каждой
    вхождения target_labels берём текст до следующей метки как значение."""
    if not text:
        return {}
    matches = list(label_pattern.finditer(text))
    result = {}
    for i, m in enumerate(matches):
        label = m.group()
        if label in target_labels and label not in result:
            start = m.end()
            end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
            value = text[start:end].strip(" ,")
            if value:
                result[label] = value
    return result

# Проверка на реальных примерах
test_examples = [
    "Онлайн-запись Тип услуги СПА-услуги, массаж Вид услуги Красота, здоровье",
    "Тип услуги автосервиса Кузовной ремонт и покраска Вид услуги Автосервис, аренда Тип услуги Автосервис",
    "Вид услуги Автосервис, аренда Тип услуги Аренда спецтехники Место оказания услуг ул. Переверткина, 22 Тип оборудования Кран-манипулятор",
]
for t in test_examples:
    print(t)
    print(" ->", extract_tags(t))
    print()

Онлайн-запись Тип услуги СПА-услуги, массаж Вид услуги Красота, здоровье
 -> {'Тип услуги': 'СПА-услуги, массаж', 'Вид услуги': 'Красота, здоровье'}

Тип услуги автосервиса Кузовной ремонт и покраска Вид услуги Автосервис, аренда Тип услуги Автосервис
 -> {'Вид услуги': 'Автосервис, аренда', 'Тип услуги': 'Автосервис'}

Вид услуги Автосервис, аренда Тип услуги Аренда спецтехники Место оказания услуг ул. Переверткина, 22 Тип оборудования Кран-манипулятор
 -> {'Вид услуги': 'Автосервис, аренда', 'Тип услуги': 'Аренда спецтехники'}



In [113]:
def get_vid_tip(text):
    tags = extract_tags(text)
    return tags.get("Вид услуги", ""), tags.get("Тип услуги", "")

# Извлекаем для объявлений в индексе валидации
items_index[["_vid", "_tip"]] = items_index["item_infm_params_text"].apply(
    lambda t: pd.Series(get_vid_tip(t))
)
# Извлекаем для val-запросов
val_queries[["_q_vid", "_q_tip"]] = val_queries["search_infm_params_text"].apply(
    lambda t: pd.Series(get_vid_tip(t))
)

# Покрытие: у какой доли val-запросов вообще нашёлся тег "Вид услуги"?
print("Доля val-запросов с непустым 'Вид услуги':", (val_queries["_q_vid"] != "").mean())
print("Доля val-запросов с непустым 'Тип услуги':", (val_queries["_q_tip"] != "").mean())

# Точность: если у запроса и его РЕАЛЬНО релевантного объявления есть оба тега,
# как часто они совпадают? (это верхняя граница пользы буста)
def check_match(row):
    rel_ids = row["relevant_item_ids"]
    rel_rows = items_index[items_index["item_id"].isin(rel_ids)]
    if len(rel_rows) == 0 or row["_q_vid"] == "":
        return np.nan
    return (rel_rows["_vid"] == row["_q_vid"]).any()

val_queries["_vid_match_check"] = val_queries.apply(check_match, axis=1)
print("\nСреди запросов, где известны оба тега -- совпадает ли 'Вид услуги' с релевантным item:")
print(val_queries["_vid_match_check"].value_counts(dropna=True, normalize=True))

Доля val-запросов с непустым 'Вид услуги': 0.6149289099526066
Доля val-запросов с непустым 'Тип услуги': 0.3164635522455428

Среди запросов, где известны оба тега -- совпадает ли 'Вид услуги' с релевантным item:
_vid_match_check
True     0.979172
False    0.020828
Name: proportion, dtype: float64


In [114]:
def top_k_candidates_boosted_v3(
    query_vecs, item_vecs, item_ids,
    query_categories, item_categories,          # microcat (как раньше)
    query_locations, item_locations,             # location (как раньше)
    query_vid, item_vid, query_tip, item_tip,     # НОВОЕ: прямые теги услуги
    category_boost=0.1, location_boost=1.0,
    vid_boost=0.0, tip_boost=0.0,
    k=50, batch_size=500,
):
    item_ids = np.asarray(item_ids)
    item_categories = np.asarray(item_categories)
    item_locations = np.asarray(item_locations)
    item_vid = np.asarray(item_vid)
    item_tip = np.asarray(item_tip)

    n_queries = query_vecs.shape[0]
    results = []
    item_vecs_T = item_vecs.T.tocsr()

    for start in range(0, n_queries, batch_size):
        end = min(start + batch_size, n_queries)
        batch_sim = (query_vecs[start:end] @ item_vecs_T).tocsr()

        for local_idx, row_idx in enumerate(range(start, end)):
            row_start, row_end = batch_sim.indptr[local_idx], batch_sim.indptr[local_idx + 1]
            cols = batch_sim.indices[row_start:row_end]
            vals = batch_sim.data[row_start:row_end].copy()

            if len(cols) == 0:
                results.append([])
                continue

            cat_match = item_categories[cols] == query_categories[row_idx]
            vals = vals + cat_match * category_boost

            loc_match = item_locations[cols] == query_locations[row_idx]
            vals = vals + loc_match * location_boost

            # буст по vid/tip -- срабатывает, только если у запроса тег известен и непустой
            q_vid = query_vid[row_idx]
            if q_vid:
                vid_match = item_vid[cols] == q_vid
                vals = vals + vid_match * vid_boost

            q_tip = query_tip[row_idx]
            if q_tip:
                tip_match = item_tip[cols] == q_tip
                vals = vals + tip_match * tip_boost

            if len(cols) > k:
                top_local = np.argpartition(vals, -k)[-k:]
            else:
                top_local = np.arange(len(cols))

            top_cols = cols[top_local]
            top_vals = vals[top_local]
            order = np.argsort(-top_vals)
            results.append(item_ids[top_cols[order]].tolist())

    return results

item_vid_arr = items_index["_vid"].values
item_tip_arr = items_index["_tip"].values
val_query_vid_arr = val_queries["_q_vid"].values
val_query_tip_arr = val_queries["_q_tip"].values

results3 = []
for vid_boost in [0.0, 0.1, 0.2, 0.3]:
    for tip_boost in [0.0, 0.1, 0.2]:
        preds = top_k_candidates_boosted_v3(
            val_query_vecs_stem2, item_vecs_stem2, items_index["item_id"].values,
            val_queries_microcat, item_microcat_idx,
            val_query_locations, item_locations_idx,
            val_query_vid_arr, item_vid_arr, val_query_tip_arr, item_tip_arr,
            category_boost=0.1, location_boost=1.0,
            vid_boost=vid_boost, tip_boost=tip_boost, k=50,
        )
        r = recall_at_k(preds, val_queries["relevant_item_ids"].tolist())
        results3.append((vid_boost, tip_boost, r))
        print(f"vid_boost={vid_boost}, tip_boost={tip_boost}: recall@50 = {r:.4f}")

best3 = max(results3, key=lambda x: x[2])
print(f"\nЛучшая комбинация: vid_boost={best3[0]}, tip_boost={best3[1]}, recall@50={best3[2]:.4f}")


vid_boost=0.0, tip_boost=0.0: recall@50 = 0.7661
vid_boost=0.0, tip_boost=0.1: recall@50 = 0.7664
vid_boost=0.0, tip_boost=0.2: recall@50 = 0.7662
vid_boost=0.1, tip_boost=0.0: recall@50 = 0.7673
vid_boost=0.1, tip_boost=0.1: recall@50 = 0.7676
vid_boost=0.1, tip_boost=0.2: recall@50 = 0.7673
vid_boost=0.2, tip_boost=0.0: recall@50 = 0.7668
vid_boost=0.2, tip_boost=0.1: recall@50 = 0.7668
vid_boost=0.2, tip_boost=0.2: recall@50 = 0.7670
vid_boost=0.3, tip_boost=0.0: recall@50 = 0.7661
vid_boost=0.3, tip_boost=0.1: recall@50 = 0.7664
vid_boost=0.3, tip_boost=0.2: recall@50 = 0.7663

Лучшая комбинация: vid_boost=0.1, tip_boost=0.1, recall@50=0.7676


In [115]:
results4 = []
for m_boost in [0.0, 0.05, 0.1]:
    preds = top_k_candidates_boosted_v3(
        val_query_vecs_stem2, item_vecs_stem2, items_index["item_id"].values,
        val_queries_microcat, item_microcat_idx,
        val_query_locations, item_locations_idx,
        val_query_vid_arr, item_vid_arr, val_query_tip_arr, item_tip_arr,
        category_boost=m_boost, location_boost=1.0,
        vid_boost=0.1, tip_boost=0.1, k=50,
    )
    r = recall_at_k(preds, val_queries["relevant_item_ids"].tolist())
    results4.append((m_boost, r))
    print(f"microcat_boost={m_boost} (при vid=0.1, tip=0.1): recall@50 = {r:.4f}")

microcat_boost=0.0 (при vid=0.1, tip=0.1): recall@50 = 0.7556
microcat_boost=0.05 (при vid=0.1, tip=0.1): recall@50 = 0.7687
microcat_boost=0.1 (при vid=0.1, tip=0.1): recall@50 = 0.7676


In [117]:
from tqdm.auto import tqdm
import time

results5 = []
combos = [
    (m, v, t)
    for m in [0.03, 0.05, 0.07]
    for v in [0.05, 0.1, 0.15]
    for t in [0.05, 0.1, 0.15]
]
print(f"Всего комбинаций: {len(combos)}")

t_start = time.time()
for i, (m_boost, v_boost, t_boost) in enumerate(tqdm(combos)):
    t0 = time.time()
    preds = top_k_candidates_boosted_v3(
        val_query_vecs_stem2, item_vecs_stem2, items_index["item_id"].values,
        val_queries_microcat, item_microcat_idx,
        val_query_locations, item_locations_idx,
        val_query_vid_arr, item_vid_arr, val_query_tip_arr, item_tip_arr,
        category_boost=m_boost, location_boost=1.0,
        vid_boost=v_boost, tip_boost=t_boost, k=50,
    )
    r = recall_at_k(preds, val_queries["relevant_item_ids"].tolist())
    results5.append((m_boost, v_boost, t_boost, r))
    print(f"[{i+1}/{len(combos)}] microcat={m_boost}, vid={v_boost}, tip={t_boost}: "
          f"recall@50={r:.4f}  ({time.time()-t0:.1f} сек)")

print(f"\nВсего времени: {(time.time()-t_start)/60:.1f} мин")
results5.sort(key=lambda x: -x[3])
print("\nТоп-10 комбинаций:")
for m, v, t, r in results5[:10]:
    print(f"microcat={m}, vid={v}, tip={t}: recall@50 = {r:.4f}")

Всего комбинаций: 27


  4%|▎         | 1/27 [05:44<2:29:17, 344.53s/it]

[1/27] microcat=0.03, vid=0.05, tip=0.05: recall@50=0.7682  (344.5 сек)


  7%|▋         | 2/27 [09:58<2:01:23, 291.34s/it]

[2/27] microcat=0.03, vid=0.05, tip=0.1: recall@50=0.7681  (254.1 сек)


...

[25/27] microcat=0.07, vid=0.15, tip=0.05: recall@50=0.7684  (163.7 сек)


 96%|█████████▋| 26/27 [1:45:37<03:10, 190.15s/it]

[26/27] microcat=0.07, vid=0.15, tip=0.1: recall@50=0.7681  (166.5 сек)


100%|██████████| 27/27 [1:48:20<00:00, 240.75s/it]

[27/27] microcat=0.07, vid=0.15, tip=0.15: recall@50=0.7681  (162.5 сек)

Всего времени: 108.3 мин

Топ-10 комбинаций:
microcat=0.05, vid=0.05, tip=0.05: recall@50 = 0.7690
microcat=0.05, vid=0.05, tip=0.1: recall@50 = 0.7689
microcat=0.05, vid=0.1, tip=0.05: recall@50 = 0.7689
microcat=0.05, vid=0.15, tip=0.05: recall@50 = 0.7687
microcat=0.05, vid=0.05, tip=0.15: recall@50 = 0.7687
microcat=0.05, vid=0.1, tip=0.1: recall@50 = 0.7687
microcat=0.07, vid=0.05, tip=0.05: recall@50 = 0.7686
microcat=0.07, vid=0.1, tip=0.05: recall@50 = 0.7685
microcat=0.05, vid=0.1, tip=0.15: recall@50 = 0.7685
microcat=0.07, vid=0.05, tip=0.1: recall@50 = 0.7685


In [118]:
# извлекаем "Вид услуги"/"Тип услуги" для корпуса и запросов бенчмарка
bench_items[["_vid", "_tip"]] = bench_items["item_infm_params_text"].apply(
    lambda t: pd.Series(get_vid_tip(t))
)
bench_queries[["_q_vid", "_q_tip"]] = bench_queries["search_infm_params_text"].apply(
    lambda t: pd.Series(get_vid_tip(t))
)
print("Покрытие 'Вид услуги' у benchmark-запросов:", (bench_queries["_q_vid"] != "").mean())
print("Покрытие 'Тип услуги' у benchmark-запросов:", (bench_queries["_q_tip"] != "").mean())

Покрытие 'Вид услуги' у benchmark-запросов: 0.333605220228385
Покрытие 'Тип услуги' у benchmark-запросов: 0.14845024469820556


In [119]:
t0 = time.time()
final_predictions_v2 = top_k_candidates_boosted_v3(
    final_query_vecs, final_item_vecs, bench_items["item_id"].values,
    bench_queries_microcat, bench_items["item_microcat_id"].values,
    bench_queries["search_location_id"].values, bench_items["item_location_id"].values,
    bench_queries["_q_vid"].values, bench_items["_vid"].values,
    bench_queries["_q_tip"].values, bench_items["_tip"].values,
    category_boost=0.05, location_boost=1.0,
    vid_boost=0.1, tip_boost=0.1, k=50,
)
print(f"Поиск занял {time.time()-t0:.1f} сек")

Поиск занял 7.8 сек


In [120]:
# те же два уровня fallback, что и раньше (переменные уже есть в ноутбуке)
preds_v2 = apply_fallback(final_predictions_v2, bench_queries["search_location_id"].values,
                          items_by_loc_bench, k=50)
preds_v2 = [global_top_fallback if len(p) == 0 else p for p in preds_v2]

answer_v2 = pd.DataFrame({
    "query_id": bench_queries["query_id"].astype(str).values,
    "answer": [" ".join(p) for p in preds_v2],
})

# проверка формата
assert len(answer_v2) == len(bench_queries) and not answer_v2["query_id"].duplicated().any()
for a in answer_v2["answer"]:
    ids = a.split(" ")
    assert 0 < len(ids) <= 50 and len(ids) == len(set(ids))
    assert all(hex16_re.match(i) and i in valid_item_ids for i in ids)

answer_v2.to_csv("answer_v2.csv", index=False, encoding="utf-8")
print("Готово: answer_v2.csv, строк:", len(answer_v2))

Fallback применён к 9 запросам
Готово: answer_v2.csv, строк: 2452


In [121]:
# 1. Координаты локаций: медиана координат всех известных объявлений (train + benchmark)
geo = pd.concat([
    train[["item_location_id", "item_latitude", "item_longitude"]],
    bench_items[["item_location_id", "item_latitude", "item_longitude"]],
])
geo["lat"] = pd.to_numeric(geo["item_latitude"], errors="coerce")
geo["lon"] = pd.to_numeric(geo["item_longitude"], errors="coerce")
loc_coords = geo.groupby("item_location_id")[["lat", "lon"]].median()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

# 2. Запросы, где релевантного объявления НЕТ в той же локации
items_by_id = items_index.set_index("item_id")
no_match = val_queries[~val_queries["_has_loc_match"]]

dists = []
for _, r in no_match.iterrows():
    loc = r["search_location_id"]
    rel_id = r["relevant_item_ids"][0]
    if loc in loc_coords.index and rel_id in items_by_id.index:
        q_lat, q_lon = loc_coords.loc[loc, "lat"], loc_coords.loc[loc, "lon"]
        i_lat = pd.to_numeric(items_by_id.loc[rel_id, "item_latitude"], errors="coerce")
        i_lon = pd.to_numeric(items_by_id.loc[rel_id, "item_longitude"], errors="coerce")
        dists.append(haversine_km(q_lat, q_lon, i_lat, i_lon))

dists = pd.Series(dists).dropna()
print(f"Запросов без location-match: {len(no_match)}, с посчитанным расстоянием: {len(dists)}")
print(dists.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]))
for km in [30, 100, 300]:
    print(f"Доля в радиусе {km} км: {(dists < km).mean():.2%}")

Запросов без location-match: 3456, с посчитанным расстоянием: 1238
count    1238.000000
mean      185.282687
std       523.966836
min         0.134010
10%         9.452587
25%        16.979453
50%        36.360631
75%        91.856640
90%       400.731063
max      6669.728806
dtype: float64
Доля в радиусе 30 км: 43.86%
Доля в радиусе 100 км: 76.66%
Доля в радиусе 300 км: 88.69%


In [122]:
# 1. Таблица «локация поиска -> где обычно выбирали объявления» (строим только по train_queries, без val)
tq = train_queries[["search_location_id", "relevant_item_ids"]].explode("relevant_item_ids")
tq = tq.merge(items_index[["item_id", "item_location_id"]],
              left_on="relevant_item_ids", right_on="item_id", how="left")
pair_counts = (tq.groupby(["search_location_id", "item_location_id"]).size()
                 .rename("n").reset_index()
                 .sort_values(["search_location_id", "n"], ascending=[True, False]))
top_locs = pair_counts.groupby("search_location_id")["item_location_id"].apply(list).to_dict()

# 2. Проверка на val-запросах без location-match
item_loc_of = items_index.set_index("item_id")["item_location_id"]
no_match = val_queries[~val_queries["_has_loc_match"]]

print("Доля no-match запросов, у локации которых нет объявлений вообще:",
      (~no_match["search_location_id"].isin(loc_coords.index)).mean())

has_map = no_match["search_location_id"].isin(top_locs)
print(f"Доля no-match запросов, для которых есть таблица соседей: {has_map.mean():.2%}")

for K in (1, 3, 5, 10):
    hits = []
    for _, r in no_match[has_map].iterrows():
        own = r["search_location_id"]
        neighbors = [l for l in top_locs[own] if l != own][:K]   # без самой локации запроса
        rel_locs = {item_loc_of[i] for i in r["relevant_item_ids"] if i in item_loc_of.index}
        hits.append(len(set(neighbors) & rel_locs) > 0)
    print(f"Релевантный item лежит в одной из топ-{K} локаций-соседей: {np.mean(hits):.2%}")

# 3. Будет ли таблица работать на бенчмарке: как часто локации его запросов встречались в train?
train_search_locs = set(train["search_location_id"].unique())
print(f"\nДоля benchmark-запросов, чья локация есть в train: "
      f"{bench_queries['search_location_id'].isin(train_search_locs).mean():.2%}")

Доля no-match запросов, у локации которых нет объявлений вообще: 0.6417824074074074
Доля no-match запросов, для которых есть таблица соседей: 99.10%
Релевантный item лежит в одной из топ-1 локаций-соседей: 57.26%
Релевантный item лежит в одной из топ-3 локаций-соседей: 68.47%
Релевантный item лежит в одной из топ-5 локаций-соседей: 72.70%
Релевантный item лежит в одной из топ-10 локаций-соседей: 78.48%

Доля benchmark-запросов, чья локация есть в train: 100.00%


In [123]:
def build_neighbor_lists(query_locs, top_locs, K=10):
    """Для каждого запроса: топ-K локаций-соседей (кроме своей собственной)"""
    out = []
    for loc in query_locs:
        out.append([l for l in top_locs.get(loc, []) if l != loc][:K])
    return out

def top_k_candidates_boosted_v4(
    query_vecs, item_vecs, item_ids,
    query_categories, item_categories,
    query_locations, item_locations,
    query_vid, item_vid, query_tip, item_tip,
    query_neighbors,
    category_boost=0.05, location_boost=1.0, vid_boost=0.1, tip_boost=0.1,
    nb_boost=0.0, K=10, k=50, batch_size=500,
):
    item_ids = np.asarray(item_ids)
    item_categories = np.asarray(item_categories)
    item_locations = np.asarray(item_locations)
    item_vid = np.asarray(item_vid)
    item_tip = np.asarray(item_tip)

    results = []
    item_vecs_T = item_vecs.T.tocsr()

    for start in range(0, query_vecs.shape[0], batch_size):
        end = min(start + batch_size, query_vecs.shape[0])
        batch_sim = (query_vecs[start:end] @ item_vecs_T).tocsr()

        for local_idx, row_idx in enumerate(range(start, end)):
            rs, re_ = batch_sim.indptr[local_idx], batch_sim.indptr[local_idx + 1]
            cols = batch_sim.indices[rs:re_]
            vals = batch_sim.data[rs:re_].copy()
            if len(cols) == 0:
                results.append([])
                continue

            vals = vals + (item_categories[cols] == query_categories[row_idx]) * category_boost
            loc_cols = item_locations[cols]
            vals = vals + (loc_cols == query_locations[row_idx]) * location_boost

            if query_vid[row_idx]:
                vals = vals + (item_vid[cols] == query_vid[row_idx]) * vid_boost
            if query_tip[row_idx]:
                vals = vals + (item_tip[cols] == query_tip[row_idx]) * tip_boost

            # НОВОЕ: буст за локации-соседи с линейным затуханием по рангу соседа
            if nb_boost > 0:
                for rank, nb_loc in enumerate(query_neighbors[row_idx]):
                    vals = vals + (loc_cols == nb_loc) * (nb_boost * (1 - rank / K))

            if len(cols) > k:
                top_local = np.argpartition(vals, -k)[-k:]
            else:
                top_local = np.arange(len(cols))
            order = np.argsort(-vals[top_local])
            results.append(item_ids[cols[top_local][order]].tolist())
    return results

# Подвыборка 3000 val-запросов (одна и та же для всех прогонов)
sub_idx = np.random.RandomState(0).choice(len(val_queries), 3000, replace=False)
sub_q = val_queries.iloc[sub_idx]
sub_neighbors = build_neighbor_lists(sub_q["search_location_id"].values, top_locs, K=10)

for nb in [0.0, 0.3, 0.6, 0.9]:
    t0 = time.time()
    preds = top_k_candidates_boosted_v4(
        val_query_vecs_stem2[sub_idx], item_vecs_stem2, items_index["item_id"].values,
        val_queries_microcat[sub_idx], item_microcat_idx,
        val_query_locations[sub_idx], item_locations_idx,
        val_query_vid_arr[sub_idx], item_vid_arr, val_query_tip_arr[sub_idx], item_tip_arr,
        sub_neighbors, nb_boost=nb,
    )
    r = recall_at_k(preds, sub_q["relevant_item_ids"].tolist())
    print(f"nb_boost={nb}: recall@50 = {r:.4f}  ({time.time()-t0:.0f} сек)")

nb_boost=0.0: recall@50 = 0.7577  (30 сек)
nb_boost=0.3: recall@50 = 0.8216  (35 сек)
nb_boost=0.6: recall@50 = 0.8232  (34 сек)
nb_boost=0.9: recall@50 = 0.8162  (35 сек)


In [124]:
# 1. Таблица соседей по ВСЕМУ train (search_location_id -> локации, где чаще выбирали объявления)
pair_counts_full = (train.groupby(["search_location_id", "item_location_id"]).size()
                    .rename("n").reset_index()
                    .sort_values(["search_location_id", "n"], ascending=[True, False]))
top_locs_full = pair_counts_full.groupby("search_location_id")["item_location_id"].apply(list).to_dict()

bench_neighbors = build_neighbor_lists(bench_queries["search_location_id"].values, top_locs_full, K=10)

# 2. Поиск с бустом за соседей
t0 = time.time()
preds_v3 = top_k_candidates_boosted_v4(
    final_query_vecs, final_item_vecs, bench_items["item_id"].values,
    bench_queries_microcat, bench_items["item_microcat_id"].values,
    bench_queries["search_location_id"].values, bench_items["item_location_id"].values,
    bench_queries["_q_vid"].values, bench_items["_vid"].values,
    bench_queries["_q_tip"].values, bench_items["_tip"].values,
    bench_neighbors,
    category_boost=0.05, location_boost=1.0, vid_boost=0.1, tip_boost=0.1,
    nb_boost=0.6, K=10, k=50,
)
print(f"Поиск занял {time.time()-t0:.0f} сек")

# 3. Fallback'и (те же, что раньше)
preds_v3 = apply_fallback(preds_v3, bench_queries["search_location_id"].values,
                          items_by_loc_bench, k=50)
preds_v3 = [global_top_fallback if len(p) == 0 else p for p in preds_v3]

# 4. Сборка и проверка формата
answer_v3 = pd.DataFrame({
    "query_id": bench_queries["query_id"].astype(str).values,
    "answer": [" ".join(p) for p in preds_v3],
})
assert len(answer_v3) == len(bench_queries) and not answer_v3["query_id"].duplicated().any()
for a in answer_v3["answer"]:
    ids = a.split(" ")
    assert 0 < len(ids) <= 50 and len(ids) == len(set(ids))
    assert all(hex16_re.match(i) and i in valid_item_ids for i in ids)

answer_v3.to_csv("answer_v3.csv", index=False, encoding="utf-8")
print("Готово: answer_v3.csv, строк:", len(answer_v3))

Поиск занял 9 сек
Fallback применён к 9 запросам
Готово: answer_v3.csv, строк: 2452


In [125]:
# Предсказания текущей лучшей схемы на той же подвыборке 3000 val-запросов (~35 сек)
preds_sub = top_k_candidates_boosted_v4(
    val_query_vecs_stem2[sub_idx], item_vecs_stem2, items_index["item_id"].values,
    val_queries_microcat[sub_idx], item_microcat_idx,
    val_query_locations[sub_idx], item_locations_idx,
    val_query_vid_arr[sub_idx], item_vid_arr, val_query_tip_arr[sub_idx], item_tip_arr,
    sub_neighbors, nb_boost=0.6,
)

item_loc_of = items_index.set_index("item_id")["item_location_id"]
rows = []
for i, (_, r) in enumerate(sub_q.iterrows()):
    own = r["search_location_id"]
    nbs = set(sub_neighbors[i])
    rel = list(r["relevant_item_ids"])
    rel_locs = [item_loc_of.get(x) for x in rel]

    if any(l == own for l in rel_locs):
        grp = "1. та же локация"
    elif any(l in nbs for l in rel_locs):
        grp = "2. локация-сосед (топ-10)"
    else:
        grp = "3. другая локация"

    recall_q = len(set(rel) & set(preds_sub[i])) / len(rel)
    rows.append((grp, recall_q))

df_br = pd.DataFrame(rows, columns=["группа", "recall"])
summary = df_br.groupby("группа")["recall"].agg(["count", "mean"])
summary["доля запросов"] = summary["count"] / summary["count"].sum()
print(summary)
print("\nОбщий recall:", df_br["recall"].mean())

                           count      mean  доля запросов
группа                                                   
1. та же локация            2394  0.890458       0.798000
2. локация-сосед (топ-10)    469  0.705224       0.156333
3. другая локация            137  0.051095       0.045667

Общий recall: 0.8231685819833189


In [126]:
# 1. Проверка гипотезы про разрыв val vs benchmark
print("Доля val-запросов, в локации которых есть объявления в индексе:",
      val_queries["search_location_id"].isin(items_index["item_location_id"]).mean())
print("Доля benchmark-запросов, в локации которых есть объявления в корпусе:",
      bench_queries["search_location_id"].isin(bench_items["item_location_id"]).mean())

# 2. Больше соседей: K = 20 и 30 (база K=10, nb_boost=0.6: recall = 0.8232)
for Kx in [20, 30]:
    nbs_k = build_neighbor_lists(sub_q["search_location_id"].values, top_locs, K=Kx)
    t0 = time.time()
    preds = top_k_candidates_boosted_v4(
        val_query_vecs_stem2[sub_idx], item_vecs_stem2, items_index["item_id"].values,
        val_queries_microcat[sub_idx], item_microcat_idx,
        val_query_locations[sub_idx], item_locations_idx,
        val_query_vid_arr[sub_idx], item_vid_arr, val_query_tip_arr[sub_idx], item_tip_arr,
        nbs_k, nb_boost=0.6, K=Kx,
    )
    r = recall_at_k(preds, sub_q["relevant_item_ids"].tolist())
    print(f"K={Kx}, nb_boost=0.6: recall@50 = {r:.4f}  ({time.time()-t0:.0f} сек)")

Доля val-запросов, в локации которых есть объявления в индексе: 0.8737305348679756
Доля benchmark-запросов, в локации которых есть объявления в корпусе: 0.8258564437194127
K=20, nb_boost=0.6: recall@50 = 0.8263  (40 сек)
K=30, nb_boost=0.6: recall@50 = 0.8260  (59 сек)


In [127]:
t0 = time.time()
preds_300 = top_k_candidates_boosted_v4(
    val_query_vecs_stem2[sub_idx], item_vecs_stem2, items_index["item_id"].values,
    val_queries_microcat[sub_idx], item_microcat_idx,
    val_query_locations[sub_idx], item_locations_idx,
    val_query_vid_arr[sub_idx], item_vid_arr, val_query_tip_arr[sub_idx], item_tip_arr,
    sub_neighbors, nb_boost=0.6, K=10, k=300,
)
print(f"{time.time()-t0:.0f} сек")

rel_sub = sub_q["relevant_item_ids"].tolist()
for kk in [50, 100, 200, 300]:
    r = recall_at_k([p[:kk] for p in preds_300], rel_sub)
    print(f"recall@{kk} = {r:.4f}")

36 сек
recall@50 = 0.8232
recall@100 = 0.8631
recall@200 = 0.8850
recall@300 = 0.8984
